<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/google_drive_summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training Runs Summary

Scans the Google Drive `mesozoic-labs/logs/` directory for all past training runs
and builds a summary table showing:

- **Species** and **algorithm** for each run
- **Key settings** (learning rate, net arch, seed, reward weights)
- **Stage pass/fail** status based on curriculum gate thresholds
- **Metrics** (avg reward, episode length, forward velocity, training time)

This notebook works in Google Colab (with Drive mounted) or locally if you
point `LOGS_DIR` at your local logs directory.

## 1. Setup & Mount Google Drive

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = (
    "COLAB_GPU" in os.environ
    or "COLAB_RELEASE_TAG" in os.environ
    or os.path.exists("/content")
)

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    LOGS_DIR = Path("/content/drive/MyDrive/mesozoic-labs/logs")
    repo_root = Path("/content/mesozoic-labs")

    # Clone the repo so `environments` package is available for imports
    if not repo_root.exists():
        import subprocess
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/kuds/mesozoic-labs.git",
             str(repo_root)],
        )
        print(f"Cloned repo to {repo_root}")
    else:
        print(f"Repo already present at {repo_root}")
else:
    # Local fallback: adjust this path if your logs are elsewhere
    repo_root = Path("..").resolve()
    LOGS_DIR = repo_root / "logs"

# Add repo root to path so `from environments.shared...` imports work
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Scanning logs directory: {LOGS_DIR}")
print(f"Directory exists: {LOGS_DIR.exists()}")

In [ ]:
import json
import re
from datetime import datetime

import numpy as np

try:
    import pandas as pd
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])
    import pandas as pd

## 2. Discover Runs and Parse Results

The training notebooks save runs to:
```
logs/<species>/<algorithm>_<YYYYMMDD_HHMMSS>/
    collected_results.csv    # rich per-stage CSV (if training notebook produced it)
    stage1/
        stage_config.json    # settings (reward weights, hyperparams, thresholds)
        evaluations.npz      # raw eval metrics (timesteps, results, ep_lengths)
        stage_summary.txt    # human-readable summary
    stage2/
        ...
    stage3/
        ...
    training_summary.txt
```

For each stage we read `stage_config.json` for the settings and curriculum
thresholds, then check `evaluations.npz` to see whether the final evaluation
metrics met those thresholds.

When the run directory contains a `collected_results.csv` (produced by the
training notebook's `save_results_csv`), we prefer its richer metrics
(forward velocity, distance traveled, success rate) over the basic
`evaluations.npz` data.

In [ ]:
def parse_run_dir_name(run_dir_name):
    """Extract algorithm and timestamp from a run directory name.

    Expected format: <algorithm>_<YYYYMMDD_HHMMSS>
    e.g. 'ppo_20260225_143021'
    """
    match = re.match(r"^(.+?)_(\d{8}_\d{6})$", run_dir_name)
    if not match:
        return run_dir_name, None
    algorithm = match.group(1).upper()
    try:
        timestamp = datetime.strptime(match.group(2), "%Y%m%d_%H%M%S")
    except ValueError:
        timestamp = None
    return algorithm, timestamp


def check_stage_passed(stage_dir, stage_config):
    """Check whether a stage passed its curriculum gate.

    Reads evaluations.npz and compares the best evaluation window against
    the thresholds from stage_config.json.

    Checks all four curriculum thresholds used by the training notebook:
    - min_avg_reward
    - min_avg_episode_length
    - min_avg_forward_vel (cannot be verified from evaluations.npz alone)
    - min_success_rate (cannot be verified from evaluations.npz alone)

    Returns (passed: bool|None, metrics: dict).
    None means we couldn't determine (missing data).
    """
    eval_path = stage_dir / "evaluations.npz"
    curriculum = stage_config.get("curriculum", {})
    min_reward = curriculum.get("min_avg_reward")
    min_length = curriculum.get("min_avg_episode_length")
    min_forward_vel = curriculum.get("min_avg_forward_vel")
    min_success_rate = curriculum.get("min_success_rate")

    metrics = {
        "best_mean_reward": None,
        "best_mean_episode_length": None,
        "last_mean_reward": None,
        "last_mean_episode_length": None,
        "timesteps_trained": None,
    }

    if not eval_path.exists():
        return None, metrics

    try:
        data = np.load(eval_path)
        results = data["results"]       # shape: (n_evals, n_episodes)
        timesteps = data["timesteps"]   # shape: (n_evals,)
    except Exception:
        return None, metrics

    if len(results) == 0:
        return None, metrics

    # Best evaluation window
    mean_rewards_per_eval = results.mean(axis=1)
    best_idx = int(mean_rewards_per_eval.argmax())
    best_mean_reward = round(float(mean_rewards_per_eval[best_idx]), 2)
    metrics["best_mean_reward"] = best_mean_reward
    metrics["timesteps_trained"] = int(timesteps[-1])

    # Last evaluation window (final metrics)
    last_mean_reward = round(float(mean_rewards_per_eval[-1]), 2)
    metrics["last_mean_reward"] = last_mean_reward

    # Episode lengths (may or may not be present)
    if "ep_lengths" in data:
        ep_lengths = data["ep_lengths"]
        metrics["best_mean_episode_length"] = round(float(ep_lengths[best_idx].mean()), 1)
        metrics["last_mean_episode_length"] = round(float(ep_lengths[-1].mean()), 1)
        best_ep_length = metrics["best_mean_episode_length"]
    else:
        best_ep_length = None

    has_any_threshold = (
        min_reward is not None
        or min_length is not None
        or min_forward_vel is not None
        or min_success_rate is not None
    )
    if not has_any_threshold:
        return None, metrics

    passed = True
    if min_reward is not None and best_mean_reward < min_reward:
        passed = False
    if min_length is not None and best_ep_length is not None and best_ep_length < min_length:
        passed = False
    # Forward velocity and success rate cannot be verified from
    # evaluations.npz alone (they require live model evaluation).
    # If collected_results.csv is available, scan_run will override
    # this pass/fail with the authoritative gate_passed value.
    if min_forward_vel is not None and min_forward_vel > 0:
        metrics["has_forward_vel_gate"] = True
    if min_success_rate is not None and min_success_rate > 0:
        metrics["has_success_rate_gate"] = True

    return passed, metrics


def parse_stage_summary_txt(stage_dir):
    """Parse stage_summary.txt for metrics not available in evaluations.npz.

    The training notebook writes this file via write_stage_summary() with
    structured lines like:
        Avg fwd vel:    0.52 +/- 0.03 m/s
        Duration:       1h 23m 45s
        Best Model Evaluation (30 episodes)
          Fwd vel:      0.48 +/- 0.05 m/s
          Success rate: 85%

    Returns a dict of extracted metrics (may be partially populated).
    """
    summary_path = stage_dir / "stage_summary.txt"
    result = {}
    if not summary_path.exists():
        return result

    try:
        text = summary_path.read_text()
    except Exception:
        return result

    # Forward velocity from final eval: "Avg fwd vel:    0.52 +/- 0.03 m/s"
    m = re.search(r"Avg fwd vel:\s+([-\d.]+)\s*\+/-\s*([-\d.]+)", text)
    if m:
        result["mean_forward_vel"] = round(float(m.group(1)), 2)
        result["std_forward_vel"] = round(float(m.group(2)), 2)

    # Best model forward velocity: "  Fwd vel:      0.48 +/- 0.05 m/s"
    m = re.search(r"Fwd vel:\s+([-\d.]+)\s*\+/-\s*([-\d.]+)", text)
    if m:
        result["best_model_fwd_vel"] = round(float(m.group(1)), 2)

    # Best model success rate: "  Success rate: 85%"
    m = re.search(r"Success rate:\s+([\d.]+)%", text)
    if m:
        result["mean_success_rate"] = round(float(m.group(1)) / 100.0, 4)

    # Best model reward: "  Reward:       1234.56 +/- 78.90"
    m = re.search(r"Best Model Evaluation.*?\n.*?Reward:\s+([-\d.]+)\s*\+/-\s*([-\d.]+)", text, re.DOTALL)
    if m:
        result["best_model_reward"] = round(float(m.group(1)), 2)

    # Best model episode length: "  Ep length:    450.0 +/- 12.3"
    m = re.search(r"Ep length:\s+([-\d.]+)\s*\+/-\s*([-\d.]+)", text)
    if m:
        result["best_model_length"] = round(float(m.group(1)), 1)

    # Duration: "Duration:       1h 23m 45s" or "Duration:       23m 45s"
    m = re.search(r"Duration:\s+(?:(\d+)h\s*)?(?:(\d+)m\s*)?(?:(\d+)s)?", text)
    if m and any(m.groups()):
        hours = int(m.group(1) or 0)
        minutes = int(m.group(2) or 0)
        seconds = int(m.group(3) or 0)
        result["training_duration_seconds"] = round(hours * 3600 + minutes * 60 + seconds, 1)

    return result


def load_collected_results_csv(run_dir):
    """Load collected_results.csv from a run directory if it exists.

    Returns a dict mapping stage number to the row dict, or None if
    the file doesn't exist.
    """
    csv_path = run_dir / "collected_results.csv"
    if not csv_path.exists():
        return None
    try:
        csv_df = pd.read_csv(csv_path)
        stage_map = {}
        for _, row in csv_df.iterrows():
            stage_map[int(row["stage"])] = row.to_dict()
        return stage_map
    except Exception:
        return None


def scan_run(species, run_dir):
    """Scan a single training run directory and return a list of row dicts.

    Data sources are consulted in priority order:
    1. ``collected_results.csv`` — richest (forward vel, distance, success
       rate, authoritative gate_passed).  Produced by the training notebook.
    2. ``stage_summary.txt`` — parsed for forward velocity, success rate,
       and training duration when the CSV is absent.
    3. ``evaluations.npz`` + ``stage_config.json`` — always used as the
       baseline for reward and episode length metrics.

    Column names follow the canonical CSV_METRIC_COLUMNS convention from
    environments.shared.reporting so that this summary CSV is consistent
    with collected_results.csv produced by sweeps and training scripts.
    """
    algorithm, timestamp = parse_run_dir_name(run_dir.name)
    algo_prefix = algorithm.lower()
    rows = []

    # Try to load the richer collected_results.csv from the run directory
    csv_data = load_collected_results_csv(run_dir)

    for stage_num in (1, 2, 3):
        stage_dir = run_dir / f"stage{stage_num}"
        if not stage_dir.is_dir():
            continue

        config_path = stage_dir / "stage_config.json"
        if config_path.exists():
            with open(config_path) as f:
                stage_config = json.load(f)
        else:
            stage_config = {}

        passed, metrics = check_stage_passed(stage_dir, stage_config)

        # Extract key settings from stage_config
        reward_weights = stage_config.get("reward_weights", {})
        hyperparams = stage_config.get("hyperparameters", {})
        curriculum = stage_config.get("curriculum", {})
        run_meta = stage_config.get("run", {})

        # Pull out the most useful settings for the summary
        lr = hyperparams.get("learning_rate")
        net_arch = hyperparams.get("policy_kwargs", {}).get("net_arch")
        if net_arch is None:
            net_arch = hyperparams.get("net_arch")
        batch_size = hyperparams.get("batch_size")
        gamma = hyperparams.get("gamma")

        # Merge richer metrics — CSV first, then stage_summary.txt fallback
        mean_forward_vel = None
        std_forward_vel = None
        mean_distance_traveled = None
        mean_success_rate = None
        training_duration_seconds = None

        if csv_data and stage_num in csv_data:
            csv_row = csv_data[stage_num]
            # Use the authoritative gate_passed from the training notebook
            csv_passed = csv_row.get("stage_passed")
            if csv_passed != "" and not pd.isna(csv_passed):
                passed = bool(csv_passed)
            # Richer metrics from live 30-episode evaluation
            _fv = csv_row.get("mean_forward_vel")
            if _fv is not None and not pd.isna(_fv):
                mean_forward_vel = round(float(_fv), 2)
            _sfv = csv_row.get("std_forward_vel")
            if _sfv is not None and not pd.isna(_sfv):
                std_forward_vel = round(float(_sfv), 2)
            _dist = csv_row.get("mean_distance_traveled")
            if _dist is not None and not pd.isna(_dist):
                mean_distance_traveled = round(float(_dist), 2)
            _sr = csv_row.get("mean_success_rate")
            if _sr is not None and not pd.isna(_sr):
                mean_success_rate = round(float(_sr), 4)
            _dur = csv_row.get("training_duration_seconds")
            if _dur is not None and not pd.isna(_dur):
                training_duration_seconds = round(float(_dur), 1)
            # Override eval metrics from CSV if present (best model eval)
            _bmr = csv_row.get("best_mean_reward")
            if _bmr is not None and not pd.isna(_bmr) and _bmr != "":
                metrics["best_mean_reward"] = round(float(_bmr), 2)
            _bml = csv_row.get("best_mean_episode_length")
            if _bml is not None and not pd.isna(_bml) and _bml != "":
                metrics["best_mean_episode_length"] = round(float(_bml), 1)
        else:
            # Fallback: parse stage_summary.txt for metrics
            txt_metrics = parse_stage_summary_txt(stage_dir)
            if mean_forward_vel is None and "mean_forward_vel" in txt_metrics:
                mean_forward_vel = txt_metrics["mean_forward_vel"]
            if std_forward_vel is None and "std_forward_vel" in txt_metrics:
                std_forward_vel = txt_metrics["std_forward_vel"]
            if mean_success_rate is None and "mean_success_rate" in txt_metrics:
                mean_success_rate = txt_metrics["mean_success_rate"]
            if training_duration_seconds is None and "training_duration_seconds" in txt_metrics:
                training_duration_seconds = txt_metrics["training_duration_seconds"]
            # Best model metrics from summary text
            if "best_model_reward" in txt_metrics and metrics["best_mean_reward"] is None:
                metrics["best_mean_reward"] = txt_metrics["best_model_reward"]
            if "best_model_length" in txt_metrics and metrics["best_mean_episode_length"] is None:
                metrics["best_mean_episode_length"] = txt_metrics["best_model_length"]

        # Use prefixed hyperparameter names and canonical metric columns
        # matching the sweep CSV conventions.
        row = {
            "species": species,
            "algorithm": algorithm,
            "library_version": stage_config.get("library_version", ""),
            "run_date": timestamp.strftime("%Y-%m-%d %H:%M") if timestamp else "",
            "run_dir": run_dir.name,
            "stage": stage_num,
            "stage_name": stage_config.get("name", ""),
            "seed": run_meta.get("seed", ""),
            "n_envs": run_meta.get("n_envs", ""),
            # Prefixed hyperparameters
            f"{algo_prefix}_learning_rate": lr,
            f"{algo_prefix}_batch_size": batch_size,
            f"{algo_prefix}_gamma": gamma,
            f"{algo_prefix}_net_arch": str(net_arch) if net_arch else "",
            "env_alive_bonus": reward_weights.get("alive_bonus"),
            "env_energy_penalty_weight": reward_weights.get("energy_penalty_weight"),
            "env_forward_vel_weight": reward_weights.get("forward_vel_weight"),
            "env_posture_weight": reward_weights.get("posture_weight"),
            # Canonical metric columns
            "best_mean_reward": metrics["best_mean_reward"],
            "best_mean_episode_length": metrics["best_mean_episode_length"],
            "last_mean_reward": metrics["last_mean_reward"],
            "last_mean_episode_length": metrics["last_mean_episode_length"],
            "mean_forward_vel": mean_forward_vel,
            "std_forward_vel": std_forward_vel,
            "mean_distance_traveled": mean_distance_traveled,
            "mean_success_rate": mean_success_rate,
            "timesteps": metrics["timesteps_trained"],
            "training_duration_seconds": training_duration_seconds,
            "reward_threshold": curriculum.get("min_avg_reward"),
            "ep_length_threshold": curriculum.get("min_avg_episode_length"),
            "forward_vel_threshold": curriculum.get("min_avg_forward_vel"),
            "success_rate_threshold": curriculum.get("min_success_rate"),
            "stage_passed": passed,
        }
        rows.append(row)

    return rows


print("Parsing functions ready.")

## 3. Scan All Runs

In [ ]:
all_rows = []

if not LOGS_DIR.exists():
    print(f"Logs directory not found: {LOGS_DIR}")
    print("Make sure Google Drive is mounted and contains mesozoic-labs/logs/.")
else:
    for species_dir in sorted(LOGS_DIR.iterdir()):
        if not species_dir.is_dir():
            continue
        species = species_dir.name
        for run_dir in sorted(species_dir.iterdir()):
            if not run_dir.is_dir():
                continue
            rows = scan_run(species, run_dir)
            all_rows.extend(rows)

df = pd.DataFrame(all_rows)
print(f"Found {len(df)} stage results across {df['run_dir'].nunique() if len(df) else 0} runs.")

## 4. Summary Table: Stage Pass/Fail by Species & Run

A pivot view showing each run as a row, with columns indicating whether
each stage passed its curriculum gate.

In [ ]:
if len(df) == 0:
    print("No runs found. Nothing to display.")
else:
    def _pass_label(val):
        if val is True:
            return "PASS"
        elif val is False:
            return "FAIL"
        return "-"

    # Build a compact pivot: one row per run, columns for each stage result
    pivot_rows = []
    for (species, run_dir), group in df.groupby(["species", "run_dir"], sort=False):
        row = {
            "species": species,
            "algorithm": group["algorithm"].iloc[0],
            "library_version": group["library_version"].iloc[0],
            "run_date": group["run_date"].iloc[0],
        }
        for _, stage_row in group.iterrows():
            sn = stage_row["stage"]
            name = stage_row["stage_name"]
            row[f"stage{sn} ({name})"] = _pass_label(stage_row["stage_passed"])
            row[f"s{sn}_reward"] = stage_row["best_mean_reward"]
        pivot_rows.append(row)

    df_pivot = pd.DataFrame(pivot_rows)
    print("=== Stage Pass/Fail Summary ===")
    display(df_pivot)

## 5. Detailed Results Table

Every stage from every run, with settings and metrics side by side.

In [ ]:
if len(df) == 0:
    print("No runs found. Nothing to display.")
else:
    # Style pass/fail for readability
    def _style_passed(val):
        if val is True:
            return "background-color: #c6efce; color: #006100"
        elif val is False:
            return "background-color: #ffc7ce; color: #9c0006"
        return ""

    display_cols = [
        "species", "algorithm", "library_version", "run_date",
        "stage", "stage_name", "stage_passed",
        "best_mean_reward", "reward_threshold",
        "best_mean_episode_length", "ep_length_threshold",
        "mean_forward_vel", "forward_vel_threshold",
        "mean_success_rate", "success_rate_threshold",
        "mean_distance_traveled",
        "last_mean_reward", "last_mean_episode_length",
        "timesteps", "training_duration_seconds",
        "seed", "n_envs",
    ]
    # Include any prefixed hyperparameter columns present in the DataFrame
    hparam_cols = sorted(c for c in df.columns if c.startswith(("ppo_", "sac_", "env_")))
    display_cols.extend(hparam_cols)
    # Only show columns that exist in df
    display_cols = [c for c in display_cols if c in df.columns]

    styled = (
        df[display_cols]
        .style
        .map(_style_passed, subset=["stage_passed"])
    )
    print("=== Detailed Stage Results ===")
    display(styled)

## 6. Settings Comparison Across Runs

Compare hyperparameters and reward weights across runs for a selected
species and stage. Useful for identifying which settings led to passes.

In [ ]:
if len(df) == 0:
    print("No runs found. Nothing to display.")
else:
    # Group by species for per-species comparison
    for species, species_df in df.groupby("species"):
        print(f"\n{'='*60}")
        print(f"  {species.upper()} - Settings vs Outcome")
        print(f"{'='*60}")

        compare_cols = [
            "algorithm", "library_version", "run_date",
            "stage", "stage_name", "stage_passed",
            "best_mean_reward", "reward_threshold",
            "mean_forward_vel", "forward_vel_threshold",
            "mean_success_rate", "success_rate_threshold",
        ]
        # Include any prefixed hyperparameter columns present
        hparam_cols = sorted(c for c in species_df.columns if c.startswith(("ppo_", "sac_", "env_")))
        compare_cols.extend(hparam_cols)
        compare_cols = [c for c in compare_cols if c in species_df.columns]
        display(species_df[compare_cols].reset_index(drop=True))

## 7. Full Settings Deep Dive (Optional)

Expand the full `stage_config.json` for any run to inspect all reward
weights and hyperparameters.

In [ ]:
def show_full_config(species, run_dir_name, stage_num):
    """Print the full stage_config.json for a specific run and stage."""
    config_path = LOGS_DIR / species / run_dir_name / f"stage{stage_num}" / "stage_config.json"
    if not config_path.exists():
        print(f"Config not found: {config_path}")
        return
    with open(config_path) as f:
        config = json.load(f)
    print(f"\n--- {species} / {run_dir_name} / stage{stage_num} ---")
    print(json.dumps(config, indent=2))


# Example: uncomment and edit to inspect a specific run
# show_full_config("trex", "ppo_20260225_143021", 1)

## 8. Export to CSV (Optional)

Save the full results table as a CSV file for further analysis.

Produces:
- `logs/runs_summary.csv` — combined results across all species
- `logs/<species>/runs_summary.csv` — per-species results file for each species found

In [ ]:
if len(df) > 0:
    from environments.shared.reporting import write_results_csv

    _fixed_columns = [
        "species", "algorithm", "library_version", "run_date", "run_dir",
        "stage", "stage_name", "seed", "n_envs",
    ]

    # Combined CSV across all species
    csv_path = write_results_csv(
        df.to_dict(orient="records"),
        LOGS_DIR / "runs_summary.csv",
        fixed_columns=_fixed_columns,
    )
    print(f"Combined summary exported to: {csv_path}")

    # Per-species CSV files
    for species, species_df in df.groupby("species"):
        species_csv_path = write_results_csv(
            species_df.to_dict(orient="records"),
            LOGS_DIR / species / "runs_summary.csv",
            fixed_columns=_fixed_columns,
        )
        print(f"  {species} summary exported to: {species_csv_path}")
else:
    print("No data to export.")